# prefill-awareness — Colab runbook

Toy end-to-end run: generate the dataset, measure the pre-SFT baseline, then
LoRA-SFT and watch whether the model learns to name the planted token.

**Pick a GPU runtime first:** Runtime > Change runtime type > T4 (or better).

The metric to watch is `loc_acc` — of all modified examples, the fraction where
the model lists exactly the right token. `detect_acc` is misleading on an
imbalanced set: a model that always answers `none` scores the majority class
while having no ability at all. Watch `tpr` too — it is 0.0 for the base model,
so anything above 0 is new capability.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
!git clone https://github.com/ydankner/training-prefill-awareness.git
%cd training-prefill-awareness
!git log --oneline | head -3

In [ ]:
# --no-deps so pip does not try to swap out Colab's CUDA build of torch.
!pip install -q --no-deps -e .
!pip install -q "transformers>=4.56" "peft>=0.13" "accelerate>=0.34" "datasets>=2.20"

# Colab ships torchao 0.10, and peft raises ImportError (rather than returning
# False) when it probes for torchao during LoRA injection unless it is >0.16.
# Nothing here uses torchao, so drop it -- upgrading risks pulling a torch build
# that does not match Colab's CUDA.
!pip uninstall -q -y torchao

## 2. Settings

T4 is Turing and has no bf16 support, so pick the dtype from the actual device.

`TOY = True` is a ~3 minute run: just enough records to look at some plants and
confirm the pipeline works. `TOY = False` is a real dataset and takes well over
an hour — generation is the long pole.

The two levers are `N_PROMPTS` and `MAX_WORDS`. `MAX_WORDS` matters more than it
looks: response length is set by the instruction rather than the decode cap, so
quartering it roughly quarters generation time.

In [ ]:
import torch

TOY = True           # ~3 min smoke run; set False for a real dataset

DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
N_PROMPTS, MAX_WORDS = (24, 120) if TOY else (300, 500)
BATCH = 8
print(f"{MODEL} | {DTYPE} | {N_PROMPTS} prompts | {MAX_WORDS} words")

## 2b. Somewhere to put the results

Colab wipes `/content` when the runtime disconnects, so artifacts are copied out
**as soon as they exist** rather than at the end of the notebook. A disconnect
during training would otherwise take the dataset with it, and the dataset is the
expensive part — the adapter can always be retrained from it.

Mounting Drive is the durable option. `snapshot()` is also safe to call when
`SAVE_TO_DRIVE = False`; it just does nothing.

In [ ]:
import os
import shutil

SAVE_TO_DRIVE = True
DEST = "/content/drive/MyDrive/prefill-awareness"

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DEST, exist_ok=True)


def snapshot(*paths):
    """Copy artifacts out of the ephemeral runtime. Call after each costly step."""
    if not SAVE_TO_DRIVE:
        return
    for path in paths:
        if not os.path.exists(path):
            continue
        dest = os.path.join(DEST, os.path.basename(path.rstrip("/")))
        if os.path.isdir(path):
            shutil.rmtree(dest, ignore_errors=True)
            shutil.copytree(path, dest)
        else:
            shutil.copy2(path, dest)
    print("in Drive:", sorted(os.listdir(DEST)))


ARTIFACTS = ("data", "runs", "baseline.json")


def download_zip(name="prefill-artifacts"):
    """Pull the artifacts down through the browser instead of Drive."""
    from google.colab import files

    staging = f"/content/{name}"
    shutil.rmtree(staging, ignore_errors=True)
    os.makedirs(staging)
    for path in ARTIFACTS:                # staged so the zip excludes the repo
        if os.path.exists(path):
            dest = os.path.join(staging, os.path.basename(path))
            (shutil.copytree if os.path.isdir(path) else shutil.copy2)(path, dest)
    shutil.make_archive(staging, "zip", staging)
    print(f"{os.path.getsize(staging + '.zip') / 1e6:.1f} MB")
    files.download(f"{staging}.zip")

## 3. Generate the dataset

Dolly's `brainstorming` split: open-ended prose, which is where the model's own
distribution has enough room for a plausible sub-1% alternative. Factoid,
classification and extraction prompts have peaked distributions and mostly fail
to yield a plant.

Sanity-check the printed counts — the clean/modified split should be near 50/50.
A heavy skew towards clean means plants are failing and the substrate is wrong.

`--greedy` decodes every position by argmax *except* the plant, which is drawn
from the sub-1% band. That makes the planted token the only place in the turn
where the model did not take its own most likely continuation, so the surprisal
signal is as clean as it can be. Under sampling, a naturally drawn token can be
as unlikely as a planted one, and the premise blurs.

Greedy interacts with `--max-orig-prob`: argmax tokens are by construction the
most likely at their position, so more positions get skipped for being too
certain and the plant yield per prompt drops. If too few plants come out, raise
`--max-orig-prob`.

In [ ]:
!python -m prefill_awareness.data_gen \
  --model {MODEL} \
  --prompts hf:databricks/databricks-dolly-15k:train \
  --categories brainstorming \
  --out data/dolly.jsonl \
  --n {N_PROMPTS} --batch-size {BATCH} --greedy \
  --max-words {MAX_WORDS} --dtype {DTYPE} --test-pct 15 --seed 0

In [ ]:
# Eyeball the plants in context before spending GPU time on training.
import json

rows = [json.loads(l) for l in open("data/dolly.jsonl", encoding="utf-8") if l.strip()]
mods = [r for r in rows if r["modification"]]
print(f"{len(rows)} records | {len(mods)} modified | "
      f"{sum(r['truncated'] for r in rows)} hit the decode cap")

for r in mods[:5]:
    m = r["modification"]
    clean, dirty = r["clean_response"], r["messages"][1]["content"]
    i = 0
    while i < min(len(clean), len(dirty)) and clean[i] == dirty[i]:
        i += 1
    print(f"\n{m['orig_token_text']!r} -> {m['new_token_text']!r}  "
          f"p_orig={m['orig_prob']:.4f} p_new={m['new_prob']:.5f} "
          f"surprisal={m['surprisal_nats']:.2f} nats")
    print("  ..." + dirty[max(0, i - 140):i + 90].replace("\n", " "))

snapshot("data")   # the costly artifact — save it before training can crash

## 4. Pre-SFT baseline

The control the whole experiment is measured against. Expect `tpr` near 0 and
`loc_acc` at 0. `format_ok` should be high — the prompt dictates the reply shape,
and format compliance is separable from actual detection ability.

In [ ]:
!python -m prefill_awareness.evaluate \
  --model {MODEL} --data data/dolly.jsonl --split test \
  --n 128 --batch-size 16 --dtype {DTYPE} --out baseline.json

## 5. LoRA SFT

Loss is masked to the final assistant turn only, so the planted tokens are never
teacher-forced. `--kl-coef` anchors the context distribution to the base model
(recovered for free by disabling the adapter); it starts at 0 and is the knob to
raise if the SFT flattens the surprisal signal it should be reading.

Evaluation runs in-loop every `--eval-every` steps, including a step-0 baseline,
and is appended to `runs/toy/eval_log.jsonl`.

In [ ]:
!python -m prefill_awareness.train \
  --model {MODEL} --data data/dolly.jsonl --out runs/toy \
  --dtype {DTYPE} --lora-r 16 --batch-size 1 --grad-accum 8 \
  --epochs 2 --lr 1e-4 --gradient-checkpointing \
  --eval-every 25 --eval-n 64 --eval-batch-size 16 --kl-coef 0.0

## 6. Results

`loc_acc` leaving 0 is the result. An always-`none` collapse (`tpr` 0, `fpr` 0)
is the expected first failure mode — if that happens, check the clean/modified
balance in the generated data before touching hyperparameters.

In [ ]:
import json

for line in open("runs/toy/eval_log.jsonl", encoding="utf-8"):
    m = json.loads(line)
    print(f"step {m['step']:>4} {m['tag']:>8} | "
          f"loc_acc {m['loc_acc']:.3f} | tpr {m['tpr']:.3f} | fpr {m['fpr']:.3f} | "
          f"detect {m['detect_acc']:.3f} | format_ok {m.get('format_ok', float('nan')):.2f}")

## 7. Take the results with you

`snapshot()` already copied the dataset to Drive after generation; this adds the
adapter, the eval log and the baseline. Everything here is small — the dataset is
a few MB of JSON and a rank-16 LoRA is tens of MB, not the full model — so
`download_zip()` through the browser is equally practical if you would rather not
use Drive.

Offline, the useful files are `runs/toy/eval_log.jsonl` (the metric timeline, one
JSON object per eval) and `data/dolly.jsonl` (every record, with the planted
token, its probabilities and surprisal under `modification`).

Worth doing as well: **File > Save a copy in GitHub**, which commits the executed
notebook — outputs included — to the private repo, so the numbers live next to
the code that produced them.

In [ ]:
snapshot(*ARTIFACTS)
# download_zip()   # uncomment to pull a zip through the browser instead